# Running PD GWAS via regenie

The intention behind this pipeline is to run a PD GWAS, so I can learn how to run GWAS end-to-end.

The data prep is described on "Ruth_PD_GWAS_data_prep.ipynb".

To make sure the pipeline worked, I should be able to see GBA, SNCA and LRRK2 popping up as strong GWAS hits (positive controls).

After running GWAS successfully, I plan to run a GWAS using tremor phenotypes.

Pipeline -adapted- from https://github.com/all-of-us/ukb-cross-analysis-demo-project/blob/main/aou_workbench_siloed_analyses/06_aou_regenie_gwas.ipynb

In [ ]:
# Start with 8 CPUs and 30gb RAM. Increase if needed. Pipeline should take about 1h to run.
!gcloud storage cp Ruth_PD_GWAS_regenie.ipynb gs://pipelines-and-files-wb-prompt-kiwi-3077/ # saving notebook in bucket

In [ ]:
from datetime import datetime
import os
import time

## 1. Setup regenie

In [ ]:
!regenie --version

In [ ]:
# OPTIONAL - It is possible to update regenie using the following code
# %%bash

# REGENIE_VERSION=v2.2.4
# rm regenie.zip
# curl -L -o regenie.zip "https://github.com/rgcgithub/regenie/releases/download/${REGENIE_VERSION}/regenie_${REGENIE_VERSION}.gz_x86_64_Linux.zip"
# unzip -o regenie.zip

# !./regenie_v2.2.4.gz_x86_64_Linux --version # --help

## 2. Finding and defining constants

In [ ]:
import os, time

# --- Inputs ---
LOCAL_PGEN_PREFIX = "array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars"

LOCAL_PGEN  = f"{LOCAL_PGEN_PREFIX}.pgen"
LOCAL_PVAR  = f"{LOCAL_PGEN_PREFIX}.pvar"
LOCAL_PSAM  = f"{LOCAL_PGEN_PREFIX}.psam"

LOCAL_PCA_EIGENVEC = "array_data/arrays_PCA.eigenvec"
LOCAL_PCA_EIGENVAL = "array_data/arrays_PCA.eigenval"

LOCAL_PHENO_COV = "PHENO_COV_DF-PD.pkl"

# Optional QC files
LOCAL_SNPLIST = "array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3.snplist"
LOCAL_PRUNE_IN = "array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3.prune.in"

# --- Outputs ---
DATESTAMP = time.strftime("%Y%m%d")
OUTPUT_FILENAME_PREFIX = "ruth_PD_GWAS"

REGENIE_OUTPUTS = f'{os.getenv("WORKSPACE_BUCKET")}/data/aou/regenie/{DATESTAMP}/'

print(REGENIE_OUTPUTS)

In [ ]:
# Check if all files were found
for f in [
    LOCAL_PGEN, LOCAL_PVAR, LOCAL_PSAM,
    LOCAL_PCA_EIGENVEC, LOCAL_PCA_EIGENVAL,
    LOCAL_PHENO_COV
]:
    print(f, os.path.exists(f))

In [ ]:
# OPTIONAL - If defining constants, modify the snippet below
# LOCAL_BGEN = os.path.basename(REMOTE_BGEN)
# LOCAL_BGEN_SAMPLE = os.path.basename(REMOTE_BGEN_SAMPLE)
# LOCAL_GWAS_PHENOTYPES = os.path.basename(REMOTE_GWAS_PHENOTYPES)
# LOCAL_STEP1_VARIANT_QC_ID = os.path.basename(REMOTE_STEP1_VARIANT_QC_ID)
# LOCAL_STEP1_VARIANT_QC_SNPLIST = os.path.basename(REMOTE_STEP1_VARIANT_QC_SNPLIST)
# LOCAL_STEP2_VARIANT_QC_ID = os.path.basename(REMOTE_STEP2_VARIANT_QC_ID)
# LOCAL_STEP2_VARIANT_QC_SNPLIST = os.path.basename(REMOTE_STEP2_VARIANT_QC_SNPLIST)

## 3. Regenie

## Regenie step 1
From https://rgcgithub.github.io/regenie/overview/:

>In the first step a subset of genetic markers are used to fit a whole genome regression model that captures a good fraction of the phenotype variance attributable to genetic effects.

In [ ]:
# Check input format
!head array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars.psam
!head array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3.snplist
!head pheno_PD_GWAS.tsv

In [ ]:
# Modify input for Regenie
#!cp arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars.psam arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars.OLD.psam

!awk 'BEGIN{OFS="\t"} NR==1{print "#FID","IID",$2; next} {print $1,$1,$2}' \
    array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars.psam \
    > array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars.fixed.psam

!head array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars.fixed.psam

In [ ]:
!mv array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars.fixed.psam array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars.psam

In [ ]:
# Regenie step 1 - Regenie separates categorical vs continuous variables
# Ideally, request 8 CPUs
!regenie \
  --step 1 \
  --bt \
  --threads 4 \
  --pgen array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars \
  --phenoFile pheno_PD_GWAS.tsv \
  --phenoColList PD \
  --covarFile pheno_PD_GWAS.tsv \
  --catCovarList SEX \
  --covarColList AGE_COV,AGE_COV_SQUARED,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
  --extract array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3.snplist \
  --bsize 1000 \
  --out PD_regenie_step1_age_sq
# --extract option not needed when input already has pruned variants

## Interpretation of output - Step 1:
Step 1 builds a genome-wide predictor of the phenotype using ridge regression.
Regenie tried several ridge penalties and chose 0.99 as best, because -logLik/N was lowest.

Rsq = 0.0175 (no age sq) / 0.0186088 (w/ age sq)
>The genome-wide predictor explains about 1.75% / 1.86% of variation in PD status. Expected for highly polygenic traits is 0.5-5%

MSE = 0.00524 / 0.00522151
>Lower is better. Mostly useful internally. Not something people report.

-logLik/N = 0.02849 / 0.0283365
>Lower is better

The run took 6.25h for 447k samples, 417k variants, 4 CPUs - which is pretty reasonable.
Next step is step 2, which is the actual GWAS.


!!! I have noticed that using --extract option in the case above was redundant, since the input under --pgen was already quality and LD-pruned. In fact, the list under --extract had more variants than the input itself, so the algorithm took the intersection.

If you ever need to subset the input, use .prune.in variants under --extract instead.

In [ ]:
!ls -lth . | head

In [ ]:
!gcloud storage cp PD_regenie_step1_age_sq* gs://pipelines-and-files-wb-prompt-kiwi-3077/

## Regenie step 2
From https://rgcgithub.github.io/regenie/overview/:

>In the second step, a larger set of genetic markers (e.g. imputed markers) are tested for association with the phenotype conditional upon the prediction from the regression model in Step 1, using a leave one chromosome out (LOCO) scheme, that avoids proximal contamination.

In [ ]:
# Regenie step 2
!regenie \
  --step 2 \
  --bt \
  --threads 4 \
  --pgen array_data/arrays_geno_hwe_maf0.01_no_blacklist_prune_r0.3_425kvars \
  --phenoFile pheno_PD_GWAS.tsv \
  --phenoColList PD \
  --covarFile pheno_PD_GWAS.tsv \
  --catCovarList SEX \
  --covarColList AGE_COV,AGE_COV_SQUARED,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
  --pred PD_regenie_step1_age_sq_pred.list \
  --bsize 400 \
  --out PD_regenie_step2_age_sq

In [ ]:
!ls -lth PD_regenie_step2_age_sq*

In [ ]:
!gcloud storage cp PD_regenie_step2_age_sq* gs://pipelines-and-files-wb-prompt-kiwi-3077/

In [ ]:
# You can add AGE_COV_SQUARED as a covariate.
# This is because the effect of age in PD risk is not constant across years.
# It is good practice to add it, but adding it doesn't change Step 1 R2 much.

## Interpretation of output - Step 2:

In [ ]:
# Inspect the output
!head PD_regenie_step2_age_sq_PD.regenie

In [ ]:
# Checking log
!grep -i "error\|warning\|case\|control\|analy" PD_regenie_step2_age_sq.log
# No fatal errors found

In [ ]:
import pandas as pd
import numpy as np

gwas = pd.read_csv("PD_regenie_step2_age_sq_PD.regenie", sep=r"\s+")

gwas.head()

In [ ]:
# Adding a p-value column
gwas["P"] = 10 ** (-gwas["LOG10P"])
gwas["NEG_LOG10P"] = gwas["LOG10P"]

In [ ]:
# Basic sanity summary
print(gwas.shape)
print(gwas.columns)

gwas[["A1FREQ", "N", "BETA", "SE", "CHISQ", "LOG10P", "P"]].describe()

# N should be close to your phenotype sample size
# A1FREQ should be between 0 and 1
# SE should not be zero
# no weird infinite values

In [ ]:
# Genomic inflation factor lambda GC
lambda_gc = np.nanmedian(gwas["CHISQ"]) / 0.4549364
lambda_gc

# ~1.00      excellent
# 1.00–1.05  usually fine
# 1.05–1.10  mild inflation
# >1.10      investigate population structure, relatedness, covariates, case-control imbalance

# With very large sample sizes and truly polygenic traits, 
# lambda can be slightly elevated even if the analysis is fine. 
# Still, it is a useful first check.

In [ ]:
# QQ plot
import matplotlib.pyplot as plt

p = gwas["P"].dropna()
p = p[(p > 0) & (p <= 1)].sort_values()

expected = -np.log10(np.arange(1, len(p) + 1) / (len(p) + 1))
observed = -np.log10(p.values)

plt.figure(figsize=(6, 6))
plt.scatter(expected, observed, s=4)
plt.plot([0, expected.max()], [0, expected.max()], linestyle="--")
plt.xlabel("Expected -log10(P)")
plt.ylabel("Observed -log10(P)")
plt.title(f"QQ plot, lambda GC = {lambda_gc:.3f}")
plt.tight_layout()
plt.savefig("PD_GWAS_QQ.png", dpi=300)
plt.show()

# A good QQ plot should mostly follow the diagonal, 
# with deviation only at the tail if there are true associations.

In [ ]:
# Finally, the Manhattan plot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Clean chromosome column
gwas = gwas.copy()
gwas["CHROM"] = gwas["CHROM"].astype(str).str.replace("chr", "", regex=False)

# Keep autosomes
gwas = gwas[gwas["CHROM"].isin([str(i) for i in range(1, 23)])].copy()
gwas["CHROM"] = gwas["CHROM"].astype(int)
gwas["GENPOS"] = pd.to_numeric(gwas["GENPOS"], errors="coerce")
gwas = gwas.dropna(subset=["CHROM", "GENPOS", "LOG10P"])

# Sort
gwas = gwas.sort_values(["CHROM", "GENPOS"])

# Create cumulative genomic position
chrom_offsets = {}
offset = 0
x_vals = []

for chrom, df_chr in gwas.groupby("CHROM"):
    chrom_offsets[chrom] = offset
    x_vals.append(df_chr["GENPOS"] + offset)
    offset += df_chr["GENPOS"].max()

gwas["BP_cum"] = pd.concat(x_vals).sort_index()

# Chromosome label positions
chrom_centers = (
    gwas.groupby("CHROM")["BP_cum"]
    .agg(lambda x: (x.min() + x.max()) / 2)
)

plt.figure(figsize=(14, 5))

for chrom, df_chr in gwas.groupby("CHROM"):
    plt.scatter(
        df_chr["BP_cum"],
        df_chr["LOG10P"],
        s=4,
        label=str(chrom)
    )

# Genome-wide significance
plt.axhline(-np.log10(5e-8), linestyle="--", linewidth=1)

# Suggestive line, optional
plt.axhline(-np.log10(1e-5), linestyle=":", linewidth=1)

plt.xticks(chrom_centers.values, chrom_centers.index)
plt.xlabel("Chromosome")
plt.ylabel("-log10(P)")
plt.title("PD GWAS Manhattan plot")
plt.tight_layout()
plt.savefig("PD_GWAS_Manhattan.png", dpi=300)
plt.show()

In [ ]:
# Extract top hits
top_hits = gwas.sort_values("P").head(20)

top_hits[
    ["CHROM", "GENPOS", "ID", "ALLELE0", "ALLELE1", "A1FREQ", "N", "BETA", "SE", "CHISQ", "LOG10P", "P"]
]

In [ ]:
top_hits.to_csv("PD_GWAS_top_hits.tsv", sep="\t", index=False)

Unfortunately, no major PD hits were found. That can be because we used a pruned SNPs list on Regenie step 2, which substantially decreased power. Because of that, I generated a new input file with all QC-passing variants (see PD GWAS prep jupyter notebook) and will re-run step 2 below.

Also, it is important to know that the analysis is only computed for subjects that are BOTH in the PLINK and pheno files.

In [ ]:
# Regenie step 2
!regenie \
  --step 2 \
  --bt \
  --threads 4 \
  --pgen array_data/arrays_geno_hwe_maf0.01_no_blacklist_fullQC \
  --phenoFile pheno_PD_GWAS.tsv \
  --phenoColList PD \
  --covarFile pheno_PD_GWAS.tsv \
  --catCovarList SEX \
  --covarColList AGE_COV,AGE_COV_SQUARED,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
  --pred PD_regenie_step1_age_sq_pred.list \
  --bsize 400 \
  --out PD_regenie_step2_age_sq_2

In [ ]:
!gcloud storage cp PD_regenie_step2_age_sq_2* gs://pipelines-and-files-wb-prompt-kiwi-3077/

In [ ]:
# Checking log
!grep -i "error\|warning\|case\|control\|analy" PD_regenie_step2_age_sq_2.log
# No fatal errors found

In [ ]:
import pandas as pd
import numpy as np

gwas = pd.read_csv("PD_regenie_step2_age_sq_2_PD.regenie", sep=r"\s+")

gwas.head()

In [ ]:
# Adding a p-value column
gwas["P"] = 10 ** (-gwas["LOG10P"])
gwas["NEG_LOG10P"] = gwas["LOG10P"]

In [ ]:
# Basic sanity summary
print(gwas.shape)
print(gwas.columns)

gwas[["A1FREQ", "N", "BETA", "SE", "CHISQ", "LOG10P", "P"]].describe()

# N should be close to your phenotype sample size
# A1FREQ should be between 0 and 1
# SE should not be zero
# no weird infinite values

In [ ]:
# Genomic inflation factor lambda GC
lambda_gc = np.nanmedian(gwas["CHISQ"]) / 0.4549364
lambda_gc

# ~1.00      excellent
# 1.00–1.05  usually fine
# 1.05–1.10  mild inflation
# >1.10      investigate population structure, relatedness, covariates, case-control imbalance

# With very large sample sizes and truly polygenic traits, 
# lambda can be slightly elevated even if the analysis is fine. 
# Still, it is a useful first check.

In [ ]:
# QQ plot
import matplotlib.pyplot as plt

p = gwas["P"].dropna()
p = p[(p > 0) & (p <= 1)].sort_values()

expected = -np.log10(np.arange(1, len(p) + 1) / (len(p) + 1))
observed = -np.log10(p.values)

plt.figure(figsize=(6, 6))
plt.scatter(expected, observed, s=4)
plt.plot([0, expected.max()], [0, expected.max()], linestyle="--")
plt.xlabel("Expected -log10(P)")
plt.ylabel("Observed -log10(P)")
plt.title(f"QQ plot, lambda GC = {lambda_gc:.3f}")
plt.tight_layout()
plt.savefig("PD_GWAS_QQ_2.png", dpi=300)
plt.show()

# A good QQ plot should mostly follow the diagonal, 
# with deviation only at the tail if there are true associations.

In [ ]:
# Finally, the Manhattan plot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Clean chromosome column
gwas = gwas.copy()
gwas["CHROM"] = gwas["CHROM"].astype(str).str.replace("chr", "", regex=False)

# Keep autosomes
gwas = gwas[gwas["CHROM"].isin([str(i) for i in range(1, 23)])].copy()
gwas["CHROM"] = gwas["CHROM"].astype(int)
gwas["GENPOS"] = pd.to_numeric(gwas["GENPOS"], errors="coerce")
gwas = gwas.dropna(subset=["CHROM", "GENPOS", "LOG10P"])

# Sort
gwas = gwas.sort_values(["CHROM", "GENPOS"])

# Create cumulative genomic position
chrom_offsets = {}
offset = 0
x_vals = []

for chrom, df_chr in gwas.groupby("CHROM"):
    chrom_offsets[chrom] = offset
    x_vals.append(df_chr["GENPOS"] + offset)
    offset += df_chr["GENPOS"].max()

gwas["BP_cum"] = pd.concat(x_vals).sort_index()

# Chromosome label positions
chrom_centers = (
    gwas.groupby("CHROM")["BP_cum"]
    .agg(lambda x: (x.min() + x.max()) / 2)
)

plt.figure(figsize=(14, 5))

for chrom, df_chr in gwas.groupby("CHROM"):
    plt.scatter(
        df_chr["BP_cum"],
        df_chr["LOG10P"],
        s=4,
        label=str(chrom)
    )

# Genome-wide significance
plt.axhline(-np.log10(5e-8), linestyle="--", linewidth=1)

# Suggestive line, optional
plt.axhline(-np.log10(1e-5), linestyle=":", linewidth=1)

plt.xticks(chrom_centers.values, chrom_centers.index)
plt.xlabel("Chromosome")
plt.ylabel("-log10(P)")
plt.title("PD GWAS Manhattan plot")
plt.tight_layout()
plt.savefig("PD_GWAS_Manhattan_2.png", dpi=300)
plt.show()

In [ ]:
# Extract top hits
top_hits = gwas.sort_values("P").head(50)

top_hits[
    ["CHROM", "GENPOS", "ID", "ALLELE0", "ALLELE1", "A1FREQ", "N", "BETA", "SE", "CHISQ", "LOG10P", "P"]
]

In [ ]:
# Search for SNCA rs356182
snca = gwas[
    (gwas["CHROM"] == 4) &
    (gwas["GENPOS"] == 89704960)
]

snca[
    ["CHROM", "GENPOS", "ID", "ALLELE0", "ALLELE1", "A1FREQ", "N", "BETA", "SE", "CHISQ", "LOG10P", "P"]
]

In [ ]:
snca_region = gwas[
    (gwas["CHROM"] == "4") &
    (gwas["GENPOS"].between(89704960 - 500_000, 89704960 + 500_000))
].copy()

snca_region.sort_values("P")[
    ["CHROM", "GENPOS", "ID", "ALLELE0", "ALLELE1", "A1FREQ", "N", "BETA", "SE", "CHISQ", "LOG10P", "P"]
].head(20)

In [ ]:
# Search for GBA E326K - 155236376
gba = gwas[
    (gwas["CHROM"] == "1") &
    (gwas["GENPOS"] == 155236376)
]

gba[
    ["CHROM", "GENPOS", "ID", "ALLELE0", "ALLELE1", "A1FREQ", "N", "BETA", "SE", "CHISQ", "LOG10P", "P"]
]

# Search for LRRK2 PD variant
lrrk2 = gwas[
    (gwas["CHROM"] == "12") &
    (gwas["GENPOS"] == 40227006)
]

lrrk2[
    ["CHROM", "GENPOS", "ID", "ALLELE0", "ALLELE1", "A1FREQ", "N", "BETA", "SE", "CHISQ", "LOG10P", "P"]
]

Once again, the GBA variants I expected did not pop-up in the Manhattan plot - and the lines above showed this happened because GBA the variants are not even there.

Jeff told me the array data is very sparse - which is ok for Step 1 but not good for Step 2.
Next step is to extract GBA E326K and N370S from the WGS dataset and then re-run Step 2.

Data will be prepared on the PD GWAS data prep pipeline.

In [ ]:
# Regenie step 2
# .pgen used has SNPs within the range between both PD variants, so we can see the context around the variants
# For this sanity check, need to lower minMAC since the variants are quite rare (MAF < 5%)
!regenie \
  --step 2 \
  --bt \
  --threads 4 \
  --minMAC 0.5 \
  --firth --approx \
  --pgen array_data/GBA_E326K_N370S_WGS \
  --phenoFile pheno_PD_GWAS.tsv \
  --phenoColList PD \
  --covarFile pheno_PD_GWAS.tsv \
  --catCovarList SEX \
  --covarColList AGE_COV,AGE_COV_SQUARED,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
  --pred PD_regenie_step1_age_sq_pred.list \
  --bsize 400 \
  --out PD_regenie_step2_GBA_E326K_N370S_WGS

In [ ]:
import pandas as pd
import numpy as np

gwas = pd.read_csv("PD_regenie_step2_GBA_E326K_N370S_WGS_PD.regenie", sep=r"\s+")

gwas.head()

# Adding a p-value column
gwas["P"] = 10 ** (-gwas["LOG10P"])
gwas["NEG_LOG10P"] = gwas["LOG10P"]

# Extract top hits
top_hits = gwas.sort_values("P").head(20)

top_hits[
    ["CHROM", "GENPOS", "ID", "ALLELE0", "ALLELE1", "A1FREQ", "N", "BETA", "SE", "CHISQ", "LOG10P", "P"]
]

In [ ]:
# Since output above was empty, need to investigate the carrier counts
!./plink2 \
  --pfile array_data/GBA_E326K_N370S_WGS \
  --freq counts \
  --out array_data/GBA_E326K_N370S_WGS_counts

!cat array_data/GBA_E326K_N370S_WGS_counts.acount

In [ ]:
# Adding coordinates
!awk 'BEGIN{OFS="\t"} !/^#/ {print $1,$2}' \
  array_data/GBA_E326K_N370S_WGS.pvar \
  > variant_coords.tsv

!tail -n +2 array_data/GBA_E326K_N370S_WGS_counts.acount > acount_no_header.tsv

!paste variant_coords.tsv acount_no_header.tsv | \
  awk 'BEGIN{OFS="\t"} BEGIN{print "COORD_CHROM","POS","#CHROM","ID","REF","ALT","ALT_CTS","OBS_CT"} {print}' \
  > array_data/GBA_E326K_N370S_WGS_counts.with_coords.tsv

!grep -i -E "155236376|155235843" array_data/GBA_E326K_N370S_WGS_counts.with_coords.tsv

We would expect about 4 E326K carriers in the PD group (1794 / 829548 * 1667 PD)... 
So that doesn't seem to give enough power to make the analysis run...

At least, the SNCA PD GWAS variant is nominally significant (p = 0.000062), as shown some cells above. This is encouraging.